In [2]:
# ================================================================
# LATIN SQUARE COMPLETION USING BACKTRACKING
# Constraint Satisfaction Problem (CSP)
#
# Features:
#   1. User input for Latin Square
#   2. Row uniqueness constraint
#   3. Column uniqueness constraint
#   4. Fixed-value constraint
#   5. Backtracking
#   6. Constraint propagation
#   7. MRV (Minimum Remaining Values)
#   8. Pruning
#   9. Solution verification
#  10. Time and Space Complexity display
# ================================================================

import time


# ------------------------------------------------
# Global statistics
# ------------------------------------------------

recursive_calls = 0
assignments = 0
backtracks = 0
pruned_branches = 0


# ------------------------------------------------
# Display the Latin Square
# ------------------------------------------------

def display_grid(grid):
    n = len(grid)

    print("\n" + "=" * (5 * n + 1))

    for row in grid:
        print("|", end="")

        for value in row:
            if value == 0:
                print("  . |", end="")
            else:
                print(f"{value:3} |", end="")

        print()

    print("=" * (5 * n + 1))


# ------------------------------------------------
# Validate the initial Latin Square
# ------------------------------------------------

def validate_initial_grid(grid):

    n = len(grid)

    # Check values
    for i in range(n):
        for j in range(n):

            value = grid[i][j]

            # 0 represents an empty cell
            if value == 0:
                continue

            # Valid symbols are 1 to N
            if value < 1 or value > n:

                print(
                    f"\nInvalid value {value} at "
                    f"row {i + 1}, column {j + 1}."
                )

                return False

    # Check duplicate values in rows
    for i in range(n):

        seen = set()

        for j in range(n):

            value = grid[i][j]

            if value != 0:

                if value in seen:

                    print(
                        f"\nDuplicate value {value} "
                        f"found in row {i + 1}."
                    )

                    return False

                seen.add(value)

    # Check duplicate values in columns
    for j in range(n):

        seen = set()

        for i in range(n):

            value = grid[i][j]

            if value != 0:

                if value in seen:

                    print(
                        f"\nDuplicate value {value} "
                        f"found in column {j + 1}."
                    )

                    return False

                seen.add(value)

    return True


# ------------------------------------------------
# Get candidates for an empty cell
# ------------------------------------------------

def get_candidates(grid, row, col):

    n = len(grid)

    # All available symbols
    symbols = set(range(1, n + 1))

    # Values already used in row
    row_values = set()

    for j in range(n):

        if grid[row][j] != 0:
            row_values.add(grid[row][j])

    # Values already used in column
    column_values = set()

    for i in range(n):

        if grid[i][col] != 0:
            column_values.add(grid[i][col])

    # Remove values that are already used
    candidates = symbols - row_values - column_values

    return candidates


# ------------------------------------------------
# Find cell using MRV
# Minimum Remaining Values
# ------------------------------------------------

def find_best_empty_cell(grid):

    n = len(grid)

    best_cell = None
    best_candidates = None

    for i in range(n):

        for j in range(n):

            if grid[i][j] == 0:

                candidates = get_candidates(grid, i, j)

                # No possible value
                if len(candidates) == 0:

                    return None, set()

                # Select the cell having
                # the smallest candidate set
                if (
                    best_candidates is None
                    or len(candidates) < len(best_candidates)
                ):

                    best_cell = (i, j)
                    best_candidates = candidates

    return best_cell, best_candidates


# ------------------------------------------------
# Constraint Propagation
# ------------------------------------------------

def constraint_propagation(grid):

    n = len(grid)

    for i in range(n):

        for j in range(n):

            if grid[i][j] == 0:

                candidates = get_candidates(grid, i, j)

                # If any empty cell has no
                # possible candidate, prune branch
                if len(candidates) == 0:

                    return False

    return True


# ------------------------------------------------
# Check whether the grid is complete
# ------------------------------------------------

def is_complete(grid):

    n = len(grid)

    for i in range(n):

        for j in range(n):

            if grid[i][j] == 0:
                return False

    return True


# ------------------------------------------------
# Verify final Latin Square
# ------------------------------------------------

def verify_solution(grid):

    n = len(grid)

    expected_symbols = set(range(1, n + 1))

    # Check rows
    for i in range(n):

        row_values = set(grid[i])

        if row_values != expected_symbols:
            return False

    # Check columns
    for j in range(n):

        column_values = set()

        for i in range(n):

            column_values.add(grid[i][j])

        if column_values != expected_symbols:
            return False

    return True


# ------------------------------------------------
# Backtracking Solver
# ------------------------------------------------

def solve_latin_square(grid):

    global recursive_calls
    global assignments
    global backtracks
    global pruned_branches

    recursive_calls += 1

    # ------------------------------------------------
    # Base Case
    # ------------------------------------------------

    if is_complete(grid):

        if verify_solution(grid):
            return True

        return False

    # ------------------------------------------------
    # Constraint Propagation
    # ------------------------------------------------

    if not constraint_propagation(grid):

        pruned_branches += 1

        return False

    # ------------------------------------------------
    # Select the best empty cell using MRV
    # ------------------------------------------------

    cell, candidates = find_best_empty_cell(grid)

    if cell is None:

        pruned_branches += 1

        return False

    row, col = cell

    # ------------------------------------------------
    # Try each candidate
    # ------------------------------------------------

    for value in sorted(candidates):

        # --------------------------------------------
        # Check row constraint
        # --------------------------------------------

        if value in grid[row]:

            pruned_branches += 1

            continue

        # --------------------------------------------
        # Check column constraint
        # --------------------------------------------

        column_values = set()

        for i in range(len(grid)):

            column_values.add(grid[i][col])

        if value in column_values:

            pruned_branches += 1

            continue

        # --------------------------------------------
        # Assign value
        # --------------------------------------------

        grid[row][col] = value

        assignments += 1

        # --------------------------------------------
        # Recursive search
        # --------------------------------------------

        if solve_latin_square(grid):

            return True

        # --------------------------------------------
        # Backtrack
        # --------------------------------------------

        grid[row][col] = 0

        backtracks += 1

    return False


# ------------------------------------------------
# Read N from user
# ------------------------------------------------

def get_order():

    while True:

        try:

            n = int(
                input(
                    "\nEnter the order N of the Latin Square: "
                )
            )

            if n <= 0:

                print("N must be greater than 0.")

            else:

                return n

        except ValueError:

            print("Please enter a valid integer.")


# ------------------------------------------------
# Read the Latin Square from user
# ------------------------------------------------

def get_grid(n):

    grid = []

    print("\nEnter the partially filled Latin Square.")
    print(f"Enter {n} values in each row.")
    print("Use 0 for an empty cell.")
    print(f"Valid symbols are 1 to {n}.")

    for i in range(n):

        while True:

            try:

                values = list(
                    map(
                        int,
                        input(
                            f"\nEnter row {i + 1}: "
                        ).split()
                    )
                )

                if len(values) != n:

                    print(
                        f"Please enter exactly {n} values."
                    )

                    continue

                grid.append(values)

                break

            except ValueError:

                print(
                    "Invalid input. "
                    "Please enter integers only."
                )

    return grid


# ------------------------------------------------
# Count empty cells
# ------------------------------------------------

def count_empty_cells(grid):

    count = 0

    for row in grid:

        for value in row:

            if value == 0:

                count += 1

    return count


# ------------------------------------------------
# Display complexity information
# ------------------------------------------------

def display_complexity():

    print("\n" + "=" * 65)
    print("COMPUTATIONAL COMPLEXITY")
    print("=" * 65)

    print("\nTime Complexity:")
    print("Worst Case : O(N^(N²))")

    print("\nExplanation:")
    print(
        "There are N² cells and up to N possible symbols "
        "for each cell in the worst case."
    )

    print(
        "Backtracking may therefore explore an exponential "
        "number of possible assignments."
    )

    print("\nSpace Complexity:")
    print("O(N²)")

    print("\nExplanation:")
    print(
        "The Latin Square requires O(N²) space, while the "
        "recursive search uses additional stack space."
    )

    print(
        "\nOptimization Used:"
    )

    print(
        "• Constraint Propagation"
    )

    print(
        "• MRV (Minimum Remaining Values)"
    )

    print(
        "• Constraint Checking"
    )

    print(
        "• Early Pruning"
    )

    print("=" * 65)


# ------------------------------------------------
# Display solver statistics
# ------------------------------------------------

def display_statistics():

    print("\n" + "=" * 65)
    print("SOLVER STATISTICS")
    print("=" * 65)

    print(
        f"Recursive Calls   : {recursive_calls}"
    )

    print(
        f"Assignments       : {assignments}"
    )

    print(
        f"Backtracks        : {backtracks}"
    )

    print(
        f"Pruned Branches   : {pruned_branches}"
    )

    print("=" * 65)


# ------------------------------------------------
# MAIN FUNCTION
# ------------------------------------------------

def main():

    global recursive_calls
    global assignments
    global backtracks
    global pruned_branches

    print("\n" + "*" * 65)

    print(
        "       LATIN SQUARE COMPLETION USING BACKTRACKING"
    )

    print(
        "              CONSTRAINT SATISFACTION PROBLEM"
    )

    print("*" * 65)

    print("\nAlgorithm Features:")
    print("1. Backtracking")
    print("2. Constraint Checking")
    print("3. Constraint Propagation")
    print("4. MRV Heuristic")
    print("5. Pruning")
    print("6. Final Solution Verification")

    # ------------------------------------------------
    # Get order
    # ------------------------------------------------

    n = get_order()

    # ------------------------------------------------
    # Get grid
    # ------------------------------------------------

    grid = get_grid(n)

    # ------------------------------------------------
    # Display input
    # ------------------------------------------------

    print("\n" + "=" * 65)
    print("INPUT LATIN SQUARE")
    print("=" * 65)

    display_grid(grid)

    # ------------------------------------------------
    # Validate initial grid
    # ------------------------------------------------

    print("\nChecking initial constraints...")

    if not validate_initial_grid(grid):

        print("\nResult:")
        print("INVALID INPUT")

        print(
            "\nThe given Latin Square violates "
            "the required constraints."
        )

        display_complexity()

        return

    print(
        "✓ Initial row and column constraints are satisfied."
    )

    # ------------------------------------------------
    # Count empty cells
    # ------------------------------------------------

    empty_cells = count_empty_cells(grid)

    print(
        f"✓ Number of empty cells: {empty_cells}"
    )

    # ------------------------------------------------
    # Start timer
    # ------------------------------------------------

    start_time = time.perf_counter()

    # ------------------------------------------------
    # Solve
    # ------------------------------------------------

    solution_found = solve_latin_square(grid)

    # ------------------------------------------------
    # Stop timer
    # ------------------------------------------------

    end_time = time.perf_counter()

    execution_time = end_time - start_time

    # ------------------------------------------------
    # Display result
    # ------------------------------------------------

    if solution_found:

        print("\n" + "*" * 65)

        print(
            "              SOLUTION FOUND"
        )

        print("*" * 65)

        print("\nCompleted Latin Square:")

        display_grid(grid)

        # ------------------------------------------------
        # Final verification
        # ------------------------------------------------

        print("\nFinal Constraint Verification:")

        if verify_solution(grid):

            print(
                "✓ Every row contains each symbol exactly once."
            )

            print(
                "✓ Every column contains each symbol exactly once."
            )

            print(
                "✓ All pre-filled values are preserved."
            )

            print(
                "✓ No constraint is violated."
            )

            print(
                "✓ Latin Square completion is VALID."
            )

        else:

            print(
                "✗ Final verification failed."
            )

    else:

        print("\n" + "*" * 65)

        print(
            "          NO VALID SOLUTION EXISTS"
        )

        print("*" * 65)

        print(
            "\nThe partially filled Latin Square "
            "cannot be completed while satisfying "
            "all row and column constraints."
        )

    # ------------------------------------------------
    # Execution time
    # ------------------------------------------------

    print("\n" + "=" * 65)
    print("EXECUTION TIME")
    print("=" * 65)

    print(
        f"Execution Time : {execution_time:.8f} seconds"
    )

    print("=" * 65)

    # ------------------------------------------------
    # Statistics
    # ------------------------------------------------

    display_statistics()

    # ------------------------------------------------
    # Complexity
    # ------------------------------------------------

    display_complexity()


# ------------------------------------------------
# Program Entry Point
# ------------------------------------------------

if __name__ == "__main__":

    main()


*****************************************************************
       LATIN SQUARE COMPLETION USING BACKTRACKING
              CONSTRAINT SATISFACTION PROBLEM
*****************************************************************

Algorithm Features:
1. Backtracking
2. Constraint Checking
3. Constraint Propagation
4. MRV Heuristic
5. Pruning
6. Final Solution Verification

Enter the order N of the Latin Square: 4

Enter the partially filled Latin Square.
Enter 4 values in each row.
Use 0 for an empty cell.
Valid symbols are 1 to 4.

Enter row 1: 1 2 3 0

Enter row 2: 3 4 0 2

Enter row 3: 2 0 4 3

Enter row 4: 4 3 2 0

INPUT LATIN SQUARE

|  1 |  2 |  3 |  . |
|  3 |  4 |  . |  2 |
|  2 |  . |  4 |  3 |
|  4 |  3 |  2 |  . |

Checking initial constraints...
✓ Initial row and column constraints are satisfied.
✓ Number of empty cells: 4

*****************************************************************
              SOLUTION FOUND
**********************************************************